# New Section

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q torch torchvision opencv-python numpy transformers pillow tqdm

In [6]:
import os
import zipfile
import glob
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

# 1. Path Setup
zip_file = "day_09.zip"
extracted_dir = "dataset_raw/day_09"
output_base_dir = "dataset_processed/day_09"

depth_dir = os.path.join(output_base_dir, "depth_maps")
hsi_dir = os.path.join(output_base_dir, "hsi_images")
hsie_dir = os.path.join(output_base_dir, "hsie_enhanced")

for d in [extracted_dir, depth_dir, hsi_dir, hsie_dir]:
    os.makedirs(d, exist_ok=True)

# 2. Extract ZIP
print(f"[INFO] Extracting {zip_file}...")
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extracted_dir)

# 3. Standard RGB to HSI Transformation
def rgb_to_hsi(image_bgr):
    img = image_bgr.astype(np.float32) / 255.0
    B, G, R = img[:, :, 0], img[:, :, 1], img[:, :, 2]

    I = (R + G + B) / 3.0

    min_rgb = np.minimum(np.minimum(R, G), B)
    S = np.zeros_like(I)
    non_zero = I > 1e-6
    S[non_zero] = 1.0 - (min_rgb[non_zero] / I[non_zero])

    num = 0.5 * ((R - G) + (R - B))
    den = np.sqrt((R - G)**2 + (R - B) * (G - B)) + 1e-6
    theta = np.arccos(np.clip(num / den, -1.0, 1.0))

    H = np.zeros_like(I)
    H[B <= G] = theta[B <= G]
    H[B > G] = 2.0 * np.pi - theta[B > G]
    H = H / (2.0 * np.pi)

    H_u8 = (H * 255).astype(np.uint8)
    S_u8 = (S * 255).astype(np.uint8)
    I_u8 = (I * 255).astype(np.uint8)

    return cv2.merge([H_u8, S_u8, I_u8]), H_u8, S_u8, I_u8

# 4. Load Depth Anything V2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Loading Depth Anything V2 on {device}...")

model_id = "depth-anything/Depth-Anything-V2-Small-hf"
image_processor = AutoImageProcessor.from_pretrained(model_id)
depth_model = AutoModelForDepthEstimation.from_pretrained(model_id).to(device)
depth_model.eval()

# 5. Process Images (Handles all extensions without duplication)
extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
image_paths = []
for ext in extensions:
    image_paths.extend(glob.glob(os.path.join(extracted_dir, "**", ext), recursive=True))

image_paths = sorted(list(set(image_paths)))
print(f"[INFO] Found {len(image_paths)} canopy images.")

for img_path in tqdm(image_paths, desc="Processing Day 04"):
    base_name = os.path.splitext(os.path.basename(img_path))[0]

    raw_bgr = cv2.imread(img_path)
    if raw_bgr is None:
        continue
    h, w = raw_bgr.shape[:2]
    raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(raw_rgb)

    # Depth estimation
    inputs = image_processor(images=pil_img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = depth_model(**inputs)
        predicted_depth = outputs.predicted_depth

    depth_map = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=(h, w),
        mode="bicubic",
        align_corners=False,
    ).squeeze().cpu().numpy()

    depth_norm = cv2.normalize(depth_map, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # HSI conversion
    hsi_img, H, S, I = rgb_to_hsi(raw_bgr)

    # HSIE Fusion
    hsie_img = cv2.merge([H, S, depth_norm])

    # Save outputs
    cv2.imwrite(os.path.join(depth_dir, f"{base_name}_depth.png"), depth_norm)
    cv2.imwrite(os.path.join(hsi_dir, f"{base_name}_hsi.png"), hsi_img)
    cv2.imwrite(os.path.join(hsie_dir, f"{base_name}_hsie.png"), hsie_img)

print("\n[SUCCESS] Day 04 Processing Completed!")

[INFO] Extracting day_09.zip...
[INFO] Loading Depth Anything V2 on cuda...


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

[INFO] Found 30 canopy images.


Processing Day 04: 100%|██████████| 30/30 [01:43<00:00,  3.46s/it]


[SUCCESS] Day 04 Processing Completed!


In [7]:
!zip -r processed_day09.zip dataset_processed/day_09/hsie_enhanced/

  adding: dataset_processed/day_09/hsie_enhanced/ (stored 0%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_120323_hsie.png (deflated 2%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_120234_hsie.png (deflated 1%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_120951_hsie.png (deflated 2%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_121138_hsie.png (deflated 1%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_120217_hsie.png (deflated 1%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_121243_hsie.png (deflated 2%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_120042_hsie.png (deflated 2%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_120259_hsie.png (deflated 2%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_121032_hsie.png (deflated 2%)
  adding: dataset_processed/day_09/hsie_enhanced/canopy_20261010_121203_hsie.png (d

In [8]:
from google.colab import files

# Triggers a direct, persistent browser download
files.download('processed_day09.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!unzip -q annotated_day_03.zip -d annotations_data

unzip:  cannot find or open annotated_day_03.zip, annotated_day_03.zip.zip or annotated_day_03.zip.ZIP.


In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

# 1. Locate annotations.xml and the HSIE images directory
# 1. Locate annotations.xml and the HSIE images directory
xml_matches = glob.glob("annotations_data/**/*.xml", recursive=True)
if not xml_matches:
    raise FileNotFoundError("Could not find annotations.xml. Ensure annotated_day_01.zip is extracted.")
xml_path = xml_matches[0]

images_dir = images_dir = "dataset_processed/day_03/hsie_enhanced"
# 2. Parse XML
tree = ET.parse(xml_path)
root = tree.getroot()

# Distinct colors for each class (BGR format)
COLOR_MAP = {
    'leaf': (0, 255, 0),             # Green
    'bud': (255, 0, 0),              # Blue
    'flower': (0, 255, 255),          # Yellow
    'pre-mature pod': (255, 165, 0), # Orange
    'mature pod': (0, 128, 255),     # Deep Orange
    'post-mature pod': (0, 0, 255)   # Red
}

# 3. Iterate through annotated images and visualize
image_tags = root.findall('image')
print(f"[INFO] Found {len(image_tags)} annotated frames in XML.\n")

for i, img_tag in enumerate(image_tags[:3]):  # Display first 3 annotated frames
    img_name = os.path.basename(img_tag.get('name'))

    # Locate matching image file
    matches = glob.glob(os.path.join(images_dir, "**", img_name), recursive=True)
    if not matches:
        matches = glob.glob(os.path.join(images_dir, "**", f"*{img_name}*"), recursive=True)

    if not matches:
        print(f"[WARNING] Image {img_name} not found in {images_dir}")
        continue

    img_path = matches[0]
    img = cv2.imread(img_path)
    overlay = img.copy()

    polygons = img_tag.findall('polygon')
    for poly in polygons:
        label = poly.get('label').lower()
        color = COLOR_MAP.get(label, (0, 255, 0))

        # Parse points string "x1,y1;x2,y2;..."
        pts_str = poly.get('points')
        pts = np.array([[float(coord) for coord in pt.split(',')] for pt in pts_str.split(';')], dtype=np.int32)
        pts = pts.reshape((-1, 1, 2))

        # Draw filled polygon and contour
        cv2.fillPoly(overlay, [pts], color)
        cv2.polylines(img, [pts], isClosed=True, color=(255, 255, 255), thickness=2)

        # Put label text
        x_min, y_min = pts[:, 0, 0].min(), pts[:, 0, 1].min()
        cv2.putText(img, label, (x_min, max(20, y_min - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Blend overlay with original image for transparency
    cv2.addWeighted(overlay, 0.35, img, 0.65, 0, img)

    # Convert BGR to RGB for matplotlib display
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 6))
    plt.imshow(img_rgb)
    plt.title(f"Frame {i}: {img_name} ({len(polygons)} Annotations)")
    plt.axis('off')
    plt.show()

FileNotFoundError: Could not find annotations.xml. Ensure annotated_day_01.zip is extracted.

In [ ]:
import os
import glob
import zipfile
import xml.etree.ElementTree as ET

# 1. Unzip processed_day02.zip if not already extracted
if not os.path.exists("dataset_processed/day_02/hsie_enhanced"):
    if os.path.exists("processed_day02.zip"):
        with zipfile.ZipFile("processed_day02.zip", "r") as zip_ref:
            zip_ref.extractall(".")
    else:
        print("Warning: processed_day02.zip not found.")

# 2. Locate annotations.xml safely
xml_matches = glob.glob("**/*.xml", recursive=True)
if not xml_matches:
    raise FileNotFoundError("annotations.xml not found! Please upload your annotation XML file to Colab.")
xml_path = xml_matches[0]

# Updated path to day_02
images_dir = "dataset_processed/day_02/hsie_enhanced"

# 3. Parse XML
tree = ET.parse(xml_path)
root = tree.getroot()
print(f"[SUCCESS] Loaded XML: {xml_path}")
print(f"[SUCCESS] Target Images Directory: {images_dir}")

# 4. Iterate and display first 3 frames with overlaid masks
for i, img_tag in enumerate(image_tags[:3]):
    img_name = os.path.basename(img_tag.get('name'))

    matches = glob.glob(os.path.join(images_dir, "**", img_name), recursive=True)
    if not matches:
        matches = glob.glob(os.path.join(images_dir, "**", f"*{img_name}*"), recursive=True)

    if not matches:
        print(f"[WARNING] Image {img_name} not found in {images_dir}")
        continue

    img_path = matches[0]
    img = cv2.imread(img_path)
    overlay = img.copy()

    polygons = img_tag.findall('polygon')
    for poly in polygons:
        label = poly.get('label').lower()
        color = COLOR_MAP.get(label, (0, 255, 0))

        pts_str = poly.get('points')
        pts = np.array([[float(coord) for coord in pt.split(',')] for pt in pts_str.split(';')], dtype=np.int32)
        pts = pts.reshape((-1, 1, 2))

        cv2.fillPoly(overlay, [pts], color)
        cv2.polylines(img, [pts], isClosed=True, color=(255, 255, 255), thickness=2)

        x_min, y_min = pts[:, 0, 0].min(), pts[:, 0, 1].min()
        cv2.putText(img, label, (x_min, max(20, y_min - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.addWeighted(overlay, 0.4, img, 0.6, 0, img)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 6))
    plt.imshow(img_rgb)
    plt.title(f"Frame {i}: {img_name} ({len(polygons)} Annotations)")
    plt.axis('off')
    plt.show()

FileNotFoundError: annotations.xml not found! Please upload your annotation XML file to Colab.